# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_after.png" alt="Image 2" style="width:300px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

First, let's connect to our SQLite database containing music store data.


In [1]:
import { SqlDatabase } from "@langchain/classic/sql_db";
import { DataSource } from "typeorm";

const datasource = new DataSource({
    type: "sqlite",
    database: "./Chinook.db", // Replace with the link to your database
});
const db = await SqlDatabase.fromDataSourceParams({
    appDataSource: datasource,
});

Define a context schema to hold customer information (first and last name) that will be available throughout the agent's execution.


In [2]:
import { z } from "zod";

const contextSchema = z.object({
    db: z.instanceof(SqlDatabase)
});
type Context = z.infer<typeof contextSchema>;

Create a tool to execute SQL queries. It supports named parameters (`:first`, `:last`) that get replaced with values from the runtime context.


In [3]:
import { tool, Runtime } from "langchain";

const executeSQL = tool(async ({ query }, runtime: Runtime<Context>) => {
    return await db.run(query);
}, {
    name: "execute_sql",
    description: "Execute a SQLite command and return results.",
    schema: z.object({
        query: z.string()
    })
});

Define the system prompt that instructs the agent how to interact with the database safely and use named parameters.


In [4]:
export const SYSTEM = `You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool \`execute_sql\` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.`

Create the agent with our tools and system prompt. Note: **no checkpointer yet**, so the agent won't remember previous conversations.


In [5]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";
import { ChatGoogle } from "@langchain/google";

const model = new ChatGoogle({
    model: "gemini-3.5-flash-lite",
});

const agent = createAgent({
    model,
    tools: [executeSQL],
    systemPrompt: SYSTEM,
    contextSchema,
})

## Repeated Queries

Ask about Frank Harris's last invoice. The agent successfully retrieves the information.


In [6]:
const stream = await agent.stream({
    messages: "This is Frank Harris, What was the total on my last invoice?",
}, {
    streamMode: "values",
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
This is Frank Harris, What was the total on my last invoice?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    thoughtSignature: "EmAKXgFpFH0TbY23xjyC3jk8npoglat6DWGMYxXshnoCjXHU3CPAG4DTLdJvwzPUHvDV6C4/stunUBYu6ecfYWn/XnFPcvPTbD7CgyzUvKOj6kZd8Gvsj5XRsj17VEj+8zE=",
    type: "functionCall",
    functionCall: {
      name: "execute_sql",
      args: { query: "SELECT name FROM sqlite_master WHERE type='table';" },
      id: "call_8539083"
    }
  }
]

┌────────────────────────────────────────────────────────────┐
│ 🔧 TOOL MESSAGE                                            │
└────────────────────────────────────────────────────────────┘
[{"name":"Album"},{"n

Now ask a follow-up question: "What were the titles?" Without memory, the agent has no idea what we're referring to and asks for clarification.


In [7]:
const stream = await agent.stream({
    messages: "What were the titles?",
}, {
    streamMode: "values",
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
What were the titles?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    thoughtSignature: "EmAKXgFpFH0TsTMxUWglIxaDagAt89ZWMNN7HjAGMmFFWrvu+Ro5i6/W1ZYl0fAII735PwSbqg11XFKPG0wRYuNglcgtTIUzymPDpxLEJePOtC5rm2RfiuH1ZISA+xaBtGs=",
    type: "functionCall",
    functionCall: {
      name: "execute_sql",
      args: { query: "SELECT name FROM sqlite_master WHERE type='table';" },
      id: "call_8059041"
    }
  }
]

┌────────────────────────────────────────────────────────────┐
│ 🔧 TOOL MESSAGE                                            │
└────────────────────────────────────────────────────────────┘
[{"name":"Album"},{"name":"Artist"},{"name":"Customer"},{"na

## Add memory

Let's fix this by adding a checkpointer. The `MemorySaver` will store conversation history so the agent can remember context across turns.


In [8]:
import * as setup from "./setup.ts";
import { createAgent } from "langchain";
import { MemorySaver } from "@langchain/langgraph";
import { ChatGoogle } from "@langchain/google";

const model = new ChatGoogle({
    model: "gemini-3.5-flash-lite",
});

const checkpointer = new MemorySaver()
const agent = createAgent({
    model,
    tools: [executeSQL],
    systemPrompt: SYSTEM,
    contextSchema,
    checkpointer
})

Ask the same question again, but this time with a `thread_id` to track the conversation.


In [9]:
const stream = await agent.stream({
    messages: "This is Frank Harris, What was the total on my last invoice?",
}, {
    streamMode: "values",
    configurable: { thread_id: "1" },
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
This is Frank Harris, What was the total on my last invoice?

┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    thoughtSignature: "EmAKXgFpFH0T0dtsGdSURbdTtDClbn8CH/fQGptuKHvBNH3qz1NpnLVxtAcNyabqTnsE+ADPwrMjb2Zo/oyjYB7KtxbUi/RfKnOofvAVBMGMzm/ehTyYfXwblxNpjYZV7yo=",
    type: "functionCall",
    functionCall: {
      name: "execute_sql",
      args: {
        query: "SELECT name, invoice_id, total, invoice_date FROM customers JOIN invoices ON customers.customer_id = invoices.customer_id WHERE customers.first_name = 'Frank' AND customers.last_name = 'Harris' ORDER BY invoice_date DESC LIMIT 5"
      },
      id: "call_5155711"
    }
  }
]

┌────────────────

Now when we ask "What were the titles?", the agent remembers the previous question about the invoice and returns the track titles! 🎵


In [10]:
const stream = await agent.stream({
    messages: "What were the titles?",
}, {
    streamMode: "values",
    configurable: { thread_id: "1" },
    context: { db }
})

for await (const step of stream) {
    displayMessage(step.messages.at(-1))
}


┌────────────────────────────────────────────────────────────┐
│ 👤 HUMAN MESSAGE                                           │
└────────────────────────────────────────────────────────────┘
What were the titles?



┌────────────────────────────────────────────────────────────┐
│ 🤖 AI MESSAGE                                              │
└────────────────────────────────────────────────────────────┘
[
  {
    thoughtSignature: "EmAKXgFpFH0TqG9hjkt4ay+aRBLd3onujusahS+qSX3/cKGiOR+9asLxeguruUicHHMnbssNJRK8tjnHtt8geZ6+CH/RpXsaGu6Dk0vUay9+qoPqSVtUFPtrweEh9yV9ObI=",
    type: "functionCall",
    functionCall: {
      name: "execute_sql",
      args: {
        query: "SELECT t.Name FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId WHERE il.InvoiceId = 374;"
      },
      id: "call_6252544"
    }
  }
]

┌────────────────────────────────────────────────────────────┐
│ 🔧 TOOL MESSAGE                                            │
└────────────────────────────────────────────────────────────┘
[{"Name":"Holier Than Thou"},{"Name":"Through The Never"},{"Name":"My Friend Of Misery"},{"Name":"The Wait"},{"Name":"Blitzkrieg"},{"Name":"So What"}]

┌────────────────────────────────────────────────────────